# Clustering Pipeline

## What problem are we solving?

**Clustering** is a form of *unsupervised learning*: we are given a pile of data points with **no labels** telling us what each one "is", and we want an algorithm to discover groups (clusters) of similar points on its own. Contrast this with *supervised learning* (e.g. classification), where every training example already comes with the correct answer.

This notebook builds three different families of clustering algorithm from the ground up, then compares them against their production-grade `scikit-learn` / `scipy` equivalents:

| Family | Idea in one sentence | Tasks |
| --- | --- | --- |
| **K-means** (hard clustering) | Every point belongs to exactly one cluster, defined by the nearest centroid. | 0–3, 10 |
| **Gaussian Mixture Model / EM** (soft clustering) | Every point has a *probability* of belonging to each cluster, modeled as a mix of overlapping bell-curves. | 4–9, 11 |
| **Agglomerative / hierarchical clustering** | Start with every point as its own cluster and repeatedly merge the closest pair, building a tree. | 12 |

## How to read each section

Every section below follows the same structure, in a "learn by example" style:

- **What it does** — a plain-English definition.
- **How it works** — the algorithm as a short numbered recipe.
- **Key lines explained** — the actual code from this project, annotated line by line.
- **Where this fits in the pipeline** — how it depends on the *previous* step and sets up the *next* one.
- **Professor's note** — a beginner-friendly aside: common pitfalls, terminology you'll see elsewhere, or a link between the toy code here and real-world tooling.
- **References** — official documentation / reputable sources to read further.

Run this notebook from the `unsupervised_learning/clustering` directory so the `__import__('N-module')` calls can find the numbered solution files.

> **Beginner tip:** you will see `__import__('0-initialize').initialize` instead of a normal `from module import function`. That's because these module filenames start with a digit (`0-initialize.py`), which is **not a legal Python identifier** — `import 0-initialize` would be a syntax error. `__import__()` is the built-in, string-based, low-level function that Python's own `import` statement calls internally, so passing it a string sidesteps the naming restriction. See the [Python official docs on `__import__()`](https://docs.python.org/3/library/functions.html#import__).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 0. Initialize K-means centroids

**What it does:** K-means needs a starting guess before it can iteratively refine cluster centers. This function produces that first guess: `k` random points, scattered somewhere plausible within the data.

**How it works:**
1. Find, for every feature/column, the minimum and maximum value across all `n` data points — this defines a *bounding box* that contains the whole dataset.
2. Draw `k` points uniformly at random inside that bounding box — one candidate centroid per cluster.
3. Return them as a `(k, d)` array.

**Key lines explained:**
```python
low, high = X.min(axis=0), X.max(axis=0)
return np.random.uniform(low, high, size=(k, d))
```
- `X.min(axis=0)` / `X.max(axis=0)`: `axis=0` means "collapse down the rows", i.e. compute one min/max *per column* (per feature), giving arrays of shape `(d,)` instead of a single global scalar.
- `np.random.uniform(low, high, size=(k, d))`: draws samples from a continuous uniform distribution on `[low, high)`. Because `low`/`high` are arrays here, NumPy *broadcasts* them across the `k` rows, so each feature gets its own valid random range instead of one range for everything.

**Where this fits in the pipeline:** this is the very first step of the whole notebook. Its output — the initial centroids `C` — is fed straight into [1-kmeans.py](./1-kmeans.py)'s iterative refinement loop.

> **Professor's note:** why uniform random, and not e.g. picking `k` real data points at random? Simplicity — but it's also the weakest choice: it can easily place a centroid far from any real data, produce empty clusters, or slow down convergence. This is exactly why `scikit-learn`'s `KMeans` defaults to a smarter scheme called `k-means++` instead (see task 10) — it's still random, but it's biased towards spreading centroids apart and near real data.

**References:**
- [NumPy — `numpy.random.uniform`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.uniform.html) (official docs)
- [scikit-learn — `KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) (see the `init` parameter for the `k-means++` alternative)

In [ ]:
initialize = __import__('0-initialize').initialize

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
plt.scatter(X[:, 0], X[:, 1], s=10)
plt.show()
print(initialize(X, 5))

## 1. K-means (from scratch)

**What it does:** partitions the `n` data points into `k` *hard* clusters — each point belongs to exactly one cluster — by alternating between "assign points to the nearest centroid" and "move each centroid to the average of its points" until nothing changes. This is the classic **Lloyd's algorithm**.

**How it works:**
1. Initialize `k` centroids (task 0).
2. **Assignment step:** assign every point to its single nearest centroid (Euclidean distance).
3. **Update step:** move each centroid to the mean position of all points currently assigned to it — this is literally where the "means" in "k-means" comes from.
4. Repeat steps 2–3 until the centroids stop moving (convergence) or a maximum number of iterations is reached.
5. If a cluster ever ends up with zero points assigned (an *empty cluster*), re-randomize its centroid rather than leaving it stuck with an undefined mean.

**Key lines explained:**
```python
distances = np.linalg.norm(X[:, np.newaxis] - C, axis=2)
clss = np.argmin(distances, axis=1)
```
- `X[:, np.newaxis]` reshapes `X` from `(n, d)` to `(n, 1, d)` so it *broadcasts* cleanly against `C`'s shape `(k, d)`. The subtraction then produces every point-minus-every-centroid difference at once, shape `(n, k, d)` — no explicit nested Python loop needed.
- `np.linalg.norm(..., axis=2)` collapses the last axis (the `d` features) into a single scalar per (point, centroid) pair: the Euclidean distance, shape `(n, k)`.
- `np.argmin(distances, axis=1)`: for each point (each row), picks the *index* of the smallest distance — that index is the assigned cluster.

```python
for j in range(k):
    mask = clss == j
    C[j] = (X[mask].mean(axis=0) if mask.any()
            else np.random.uniform(low, high))
```
- `mask = clss == j` is a boolean array selecting every point currently assigned to cluster `j`.
- If at least one point is assigned (`mask.any()`), the new centroid is simply the mean of those points.
- Otherwise (an empty cluster), draw a brand new random centroid — same trick as task 0 — instead of leaving a `NaN` from averaging zero points.

```python
if np.array_equal(C, C_prev):
    break
```
Convergence check: if not a single centroid moved this iteration, further iterations would be identical — stop early.

**Where this fits in the pipeline:** this function is a reusable building block called by [3-optimum.py](./3-optimum.py) (which runs it once per candidate `k` for the elbow method) and by [4-initialize.py](./4-initialize.py) (which uses its output as a smart starting guess for the GMM's means, instead of pure random Gaussians).

> **Professor's note:** K-means is mathematically *guaranteed* to converge (variance can only decrease or stay flat each iteration), but only to a **local optimum** — the result depends on where you started. That's exactly why, in practice, people run K-means several times with different random initializations and keep the run with the lowest variance; `scikit-learn`'s `KMeans` automates this with its `n_init` parameter (see task 10).

**References:**
- [NumPy — `numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html) (official docs — default behavior is the Euclidean/L2 norm)
- [scikit-learn — `KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) (documents Lloyd's algorithm and its `O(k·n·T)` average complexity)

In [ ]:
kmeans = __import__('1-kmeans').kmeans

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
C, clss = kmeans(X, 5)
print(C)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(C[:, 0], C[:, 1], s=50, marker='*', c=list(range(5)))
plt.show()

## 2. Intra-cluster variance

**What it does:** produces a single number that answers "how tightly packed are these clusters?" — the sum of squared distances from every point to its own (nearest) centroid. Lower means tighter, more compact clusters.

**How it works:**
1. For every point, compute its squared distance to *every* centroid.
2. Keep only the smallest of those distances per point — this implicitly recovers "which cluster does this point belong to", without needing an explicit label array.
3. Sum those minimum squared distances across every point.

**Key lines explained:**
```python
distances_sq = np.sum((X[:, np.newaxis] - C) ** 2, axis=2)
return np.sum(np.min(distances_sq, axis=1))
```
- Same broadcasting trick as task 1 (`X[:, np.newaxis] - C` → shape `(n, k, d)`), but here we square and sum manually instead of calling `np.linalg.norm` — since we only need *squared* distance, this skips an unnecessary square-root-then-square round trip.
- `np.min(distances_sq, axis=1)`: the distance from each point to its nearest centroid, shape `(n,)`.
- The outer `np.sum`: adds all of those up into the final scalar.

**Where this fits in the pipeline:** this is the metric [3-optimum.py](./3-optimum.py) uses to compare K-means solutions across different values of `k`. It's the same quantity `scikit-learn` calls **inertia** (see `KMeans.inertia_`).

> **Professor's note:** this quantity is also called the **within-cluster sum of squares (WCSS)**. It can *only* decrease (or stay flat) as `k` grows — in the extreme, `k = n` (one cluster per point) drives it all the way to `0`. That means you can never pick the "best" `k` by minimizing variance alone; you need a heuristic that also penalizes extra clusters, which is exactly what the elbow method (task 3) and BIC (task 9) are for.

**References:**
- General clustering-evaluation concept — see the "inertia" / within-cluster sum of squares terminology in [scikit-learn's clustering guide](https://scikit-learn.org/stable/modules/clustering.html#k-means).

In [ ]:
variance = __import__('2-variance').variance

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
C = np.random.randint(0, 4, X.shape[0])
print(variance(X, C))

## 3. Optimum number of clusters (elbow method)

**What it does:** since we rarely know the "right" `k` in advance, this runs K-means for a whole range of candidate `k` values and gives us the data needed to visually pick the best one — the classic **elbow method**.

**How it works:**
1. For every candidate `k` from `kmin` to `kmax`, run full K-means (task 1) to get centroids and cluster assignments.
2. Compute that `k`'s total intra-cluster variance (task 2).
3. Express each variance as its *drop* relative to the variance at `kmin` (the smallest `k` tested) — this "delta variance" starts at `0` by definition and grows as `k` increases.
4. Plot delta-variance against `k`. The **elbow** — the point after which adding more clusters barely helps anymore — is a good candidate for the true number of clusters.

**Key lines explained:**
```python
for k in range(kmin, kmax + 1):
    C, clss = kmeans(X, k, iterations)
    results.append((C, clss))
    variances.append(variance(X, C))
d_vars = list(variances[0] - np.array(variances))
```
- The loop runs a *complete* K-means fit once per candidate `k` — this is why the elbow method can get slow on large datasets: it's `O(kmax - kmin)` full K-means runs, not one cheap pass.
- `variances[0]` is the variance at `k = kmin` (fewest clusters, highest variance) — it's used as the baseline.
- `variances[0] - np.array(variances)`: one vectorized subtraction computing "how much variance we've recovered by adding more clusters," for every `k` at once.

**Where this fits in the pipeline:** this is the model-selection companion to K-means (tasks 0–1), exactly the way task 9 (BIC) is the model-selection companion to the GMM built in tasks 4–8. Notice the two families use *different* criteria for picking `k`: a raw, visually-inspected variance drop here, versus a statistically grounded score there.

> **Professor's note:** the elbow method is popular because it's simple and visual — but the "elbow" is frequently ambiguous or entirely absent, which is a well-documented weakness. More rigorous alternatives exist, such as the silhouette score or the gap statistic for K-means, or BIC/AIC for probabilistic models like GMMs (see task 9).

**References:**
- [Wikipedia — Elbow method (clustering)](https://en.wikipedia.org/wiki/Elbow_method_(clustering))

In [ ]:
optimum_k = __import__('3-optimum').optimum_k

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
results, d_var = optimum_k(X, kmax=10)
print(d_var)
for i, (C, c) in enumerate(results):
    print("{}: {}".format(i + 1, c))
plt.plot(range(1, 11), d_var, 'bo-')
plt.xlabel('Clusters')
plt.ylabel('Delta variance')
plt.title('Optimizing number of clusters')
plt.show()

## 4. Initialize GMM parameters

**What it does:** a **Gaussian Mixture Model (GMM)** is a *soft* clustering model — instead of saying a point belongs 100% to one cluster, it assigns each point a probability of belonging to each of `k` overlapping Gaussian ("bell curve") distributions. Before the EM algorithm (tasks 6–8) can fit those Gaussians, it needs a starting guess for three sets of parameters: `pi` (mixing weights/priors), `m` (means), and `S` (covariances).

**How it works:**
1. Run ordinary K-means (task 1) on the data to get `k` hard clusters almost instantly — reuse its centroids as a smart starting guess for the Gaussian means `m`.
2. Give every cluster an equal prior probability, `pi[j] = 1/k` — before EM has run even once, there's no reason to think any cluster is more likely than another.
3. Give every cluster's covariance matrix `S[j]` a plain **identity matrix** — i.e. assume every cluster starts out perfectly circular/spherical with unit variance in every direction. EM will reshape these into whatever ellipses actually fit the data as it iterates.

**Key lines explained:**
```python
m, _ = kmeans(X, k)
pi = np.ones(k) / k
S = np.tile(np.eye(d), (k, 1, 1))
```
- `np.eye(d)`: the `d × d` identity matrix (`1`s on the diagonal, `0`s elsewhere).
- `np.tile(..., (k, 1, 1))`: stacks that identity matrix `k` times along a brand-new leading axis, producing shape `(k, d, d)` — one starting covariance matrix per cluster.

**Where this fits in the pipeline:** this function is the bridge between the "hard" clustering world (K-means, tasks 0–3) and the "soft"/probabilistic world (GMM/EM, tasks 4–9) — notice it literally imports and calls [1-kmeans.py](./1-kmeans.py)'s `kmeans`. Its output `(pi, m, S)` is exactly the tuple the E-step (task 6) and the full EM loop (task 8) expect as their starting point.

> **Professor's note:** why not initialize `m` with pure random points, the way task 0 does for K-means? Because K-means already produces a reasonable rough clustering almost for free; starting EM near a decent solution converges faster and is far less likely to get stuck in a poor local optimum than starting from Gaussians scattered uniformly across the whole space.

**References:**
- [NumPy — `numpy.eye`](https://numpy.org/doc/stable/reference/generated/numpy.eye.html) / [`numpy.tile`](https://numpy.org/doc/stable/reference/generated/numpy.tile.html) (official docs)

In [ ]:
initialize_gmm = __import__('4-initialize').initialize

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
print(pi)
print(m)
print(S)

## 5. Gaussian probability density function (PDF)

**What it does:** before we can ask "how likely is this point under cluster `j`'s Gaussian?", we need the actual multivariate Gaussian (normal) probability density function, computed for every point in the dataset at once.

**The formula:**

$$f(x) = \frac{1}{(2\pi)^{d/2}\sqrt{|\Sigma|}} \exp\left(-\frac{1}{2}(x-\mu)^T \Sigma^{-1} (x-\mu)\right)$$

where $\mu$ is the mean vector (`m`), $\Sigma$ is the covariance matrix (`S`), $d$ is the number of dimensions, and $(x-\mu)^T \Sigma^{-1}(x-\mu)$ is the squared **Mahalanobis distance** — informally, "how many standard deviations away is `x` from the mean, accounting for correlations between features."

**How it works:**
1. Convert `m`/`S` to NumPy arrays defensively (the function should also accept plain Python lists — see the beginner note below).
2. Compute $\Sigma^{-1}$ and $|\Sigma|$ — per-cluster constants that don't depend on `x`.
3. Compute `X - mu` for every point at once (broadcasting).
4. Compute the Mahalanobis term for every point in one shot.
5. Plug into the Gaussian formula, then clip the result to a tiny positive floor (`1e-300`) so it never becomes a literal `0.0`.

**Key lines explained:**
```python
S_inv = np.linalg.inv(S)
det = np.linalg.det(S)
diff = X - m
coeff = 1 / (((2 * np.pi) ** (d / 2)) * np.sqrt(det))
mahal = np.sum((diff @ S_inv) * diff, axis=1)
return np.maximum(coeff * np.exp(-0.5 * mahal), 1e-300)
```
- `np.linalg.inv` / `np.linalg.det`: matrix inverse and determinant, from NumPy's linear-algebra module.
- `diff @ S_inv`: matrix multiplication using Python's dedicated `@` operator (introduced by [PEP 465](https://peps.python.org/pep-0465/) in Python 3.5) between the `(n, d)` difference matrix and the `(d, d)` inverse covariance.
- `np.maximum(..., 1e-300)`: an elementwise floor — a very common numerical-stability trick throughout machine learning code.

**Where this fits in the pipeline:** this is the mathematical core every later GMM step depends on. It's called once per cluster inside the E-step (task 6) to answer "how likely does cluster `j`'s bell curve think this point is?"

> **Professor's note:** `1e-300` isn't an ML-specific magic number — it's simply far above `0` but far below any realistic density value, chosen so that taking `log()` of it later (task 6) still gives a large-but-finite negative number instead of `-inf`, which would otherwise poison the whole log-likelihood sum with `NaN`/`-inf` and crash the EM iteration. Also note: this project's `pdf` originally required `m`/`S` to already be `numpy.ndarray` and rejected plain Python lists outright — that's why it now calls `np.asarray()` first, so it degrades gracefully instead of returning `None` unexpectedly.

**References:**
- [NumPy — Linear algebra (`numpy.linalg`)](https://numpy.org/doc/stable/reference/routines.linalg.html) (official docs, covers `inv`/`det`)
- Wikipedia — [Mahalanobis distance](https://en.wikipedia.org/wiki/Mahalanobis_distance), [Multivariate normal distribution](https://en.wikipedia.org/wiki/Multivariate_normal_distribution)

In [ ]:
pdf = __import__('5-pdf').pdf

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
P = pdf(X, [30, 40], [[75, 5], [5, 75]])
print(P)
print("shape:", P.shape)
print("min value:", P.min())

## 6. Expectation step (E-step)

**What it does:** given the model's current guesses (`pi`, `m`, `S`), estimate — for every point and every cluster — the probability that this point belongs to this cluster. In EM terminology this is the **responsibility** that cluster `j` takes for point `i`.

**The formula** (this is just Bayes' theorem):

$$g_{j,i} = \frac{\pi_j \, \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}{\sum_{l=1}^{k} \pi_l \, \mathcal{N}(x_i \mid \mu_l, \Sigma_l)}$$

prior ($\pi_j$) times likelihood ($\mathcal{N}$, computed via task 5's `pdf`), divided by the total evidence (the sum over every cluster), gives the posterior probability.

**How it works:**
1. For every cluster `j`, compute the *weighted* likelihood `pi[j] * pdf(X, m[j], S[j])` for every point — an unnormalized "vote" from cluster `j` for every point.
2. Sum these votes across clusters, per point, to get the total evidence (the denominator of Bayes' rule).
3. The dataset's **log-likelihood** — the single number EM is trying to maximize, iteration after iteration — is the sum of the logs of those totals.
4. Normalize: divide every cluster's vote by the total, so each point's `k` responsibilities sum to exactly `1`.

**Key lines explained:**
```python
g = np.zeros((k, n))
for j in range(k):
    g[j] = pi[j] * pdf(X, m[j], S[j])
total = g.sum(axis=0)
log_l = np.sum(np.log(total))
g /= total
```
- The loop runs over `k` (the number of clusters), not `n` (the number of points) — `k` is usually small while `n` can be huge, so looping over `k` and vectorizing over `n` inside `pdf` is the efficient choice.
- `g.sum(axis=0)`: sums *down each column* — i.e. across clusters, for every point — giving one total per point, shape `(n,)`.
- `g /= total`: in-place elementwise division; `total`'s shape `(n,)` broadcasts against `g`'s shape `(k, n)`, so every row of `g` is divided by the same per-point total.

**Where this fits in the pipeline:** this is the "E" in "EM". Its output `g` (the responsibility matrix) is exactly what the M-step (task 7) needs to recompute better priors/means/covariances, and `log_l` is exactly what the full EM loop (task 8) watches to detect convergence.

> **Professor's note:** `g` has shape `(k, n)` here — one row *per cluster*. A very common bug when implementing EM by hand is transposing this matrix, since `(k, n)` and `(n, k)` are both plausible-looking shapes for the same data — keep the convention straight, or better, print `g.shape` while debugging.

**References:**
- [Wikipedia — Expectation–maximization algorithm](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm)
- Wikipedia — [Bayes' theorem](https://en.wikipedia.org/wiki/Bayes%27_theorem)

In [ ]:
initialize_gmm = __import__('4-initialize').initialize
expectation = __import__('6-expectation').expectation

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
g, l = expectation(X, pi, m, S)
print(g)
print(np.sum(g, axis=0))
print(l)

## 7. Maximization step (M-step)

**What it does:** given the E-step's responsibilities (`g`), recompute the model's parameters (`pi`, `m`, `S`) to be the ones that best explain the data, weighted by those responsibilities. This is where the actual "learning" happens each iteration.

**The formulas:**

$$N_j = \sum_{i=1}^n g_{j,i} \qquad \pi_j = \frac{N_j}{n} \qquad \mu_j = \frac{1}{N_j}\sum_{i=1}^n g_{j,i}\, x_i \qquad \Sigma_j = \frac{1}{N_j}\sum_{i=1}^n g_{j,i}\,(x_i-\mu_j)(x_i-\mu_j)^T$$

$N_j$ is the "effective number of points" softly assigned to cluster `j` — a weighted count, generally not an integer, since responsibilities are fractional.

**How it works:**
1. `N_j` = sum of responsibilities for cluster `j` across all points — that cluster's effective population.
2. New prior `pi[j]` = that population's share of the whole dataset.
3. New mean `m[j]` = a weighted average of all points, weighted by their responsibility to cluster `j` — points strongly believed to belong to cluster `j` pull its mean towards them more.
4. New covariance `S[j]` = the weighted scatter of points around the new mean — the familiar sample-covariance formula, but weighted by responsibility instead of a hard 0/1 cluster membership.

**Key lines explained:**
```python
N = g.sum(axis=1)
pi = N / n
m = (g @ X) / N[:, np.newaxis]
for j in range(k):
    diff = X - m[j]
    S[j] = (g[j, :, np.newaxis] * diff).T @ diff / N[j]
```
- `g.sum(axis=1)`: sums across points (`axis=1`) for every cluster — computes every $N_j$ at once, shape `(k,)`.
- `g @ X`: a single matrix multiplication, `(k, n) @ (n, d) → (k, d)`, that computes *all* `k` weighted sums-of-points at once — row `j` of the result is $\sum_i g_{j,i} x_i$.
- `N[:, np.newaxis]`: reshapes `(k,)` to `(k, 1)` so it broadcasts correctly against the `(k, d)` numerator, dividing every row `j` by its own $N_j$.
- Inside the loop, `g[j, :, np.newaxis] * diff` scales every row of the `(n, d)` difference matrix by that point's responsibility to cluster `j`; `.T @ diff` then performs the weighted outer-product sum in one line, producing the `(d, d)` weighted scatter/covariance matrix.

**Where this fits in the pipeline:** this is the "M" in "EM". Its output `(pi, m, S)` either feeds *back* into the next E-step (task 6) — the back-and-forth automated by task 8 — or, on the final iteration, becomes the pipeline's finished, fitted model.

> **Professor's note:** tasks 6 and 7 together are a textbook example of **vectorization**: every operation that *could* have been a nested Python `for` loop over `n` (which might be tens of thousands of points) is instead a NumPy array operation, executed as compiled, low-level code rather than the slower Python interpreter loop. This habit is usually worth a 10–100x speedup, and it's worth building early in your ML career.

**References:**
- Python — [PEP 465, the `@` matrix multiplication operator](https://peps.python.org/pep-0465/)
- [Wikipedia — Expectation–maximization algorithm](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm) (see the "Gaussian mixture" worked example)

In [ ]:
initialize_gmm = __import__('4-initialize').initialize
expectation = __import__('6-expectation').expectation
maximization = __import__('7-maximization').maximization

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
g, _ = expectation(X, pi, m, S)
pi, m, S = maximization(X, g)
print(pi)
print(m)
print(S)

## 8. Full EM algorithm

**What it does:** the complete loop that alternates the E-step and M-step, tying tasks 4, 6, and 7 together, until the model's log-likelihood stops improving (or a maximum number of iterations is reached).

**How it works:**
1. Initialize `pi`, `m`, `S` (task 4).
2. Repeat, up to `iterations` times:
   - **E-step** (task 6): compute responsibilities `g` and the current log-likelihood.
   - **Check for convergence:** if the log-likelihood barely changed since the previous iteration (within `tol`), stop early — the model has settled.
   - **M-step** (task 7): recompute `pi`, `m`, `S` from `g`.
3. Optionally print progress (`verbose=True`) every 10 iterations, and always print the final iteration.

**Key lines explained:**
```python
l_prev = 0
for i in range(iterations):
    g, log_l = expectation(X, pi, m, S)
    if i > 0 and abs(log_l - l_prev) <= tol:
        break
    l_prev = log_l
    pi, m, S = maximization(X, g)
```
- The `i > 0` guard prevents stopping on the very first iteration, before there's a meaningful `l_prev` to compare against (it starts as a placeholder `0`).
- Notice the order: the E-step runs, *then* convergence is checked using this iteration's likelihood, and *only if we're not stopping* does the M-step run. This guarantees the function returns a `g` that is consistent with the `pi, m, S` it reports — both computed from the *same* E-step.

**Where this fits in the pipeline:** this is the single entry point you'd actually call to fit a full custom GMM — everything in tasks 4–7 is an internal building block for this one function. Its output feeds directly into task 9 (BIC), which calls it once per candidate `k` to decide how many clusters best fit the data — the GMM's analogue of task 3's elbow-method loop over K-means.

> **Professor's note:** the log-likelihood is mathematically *guaranteed* to increase (or stay flat) every single EM iteration. If you ever implement EM yourself and see it *decrease*, that's a strong signal of a bug in your E-step or M-step math — not bad luck. It's a great, nearly-free built-in sanity check (try setting `verbose=True` in the cell below and watch the numbers only ever go up).

**References:**
- [Wikipedia — Expectation–maximization algorithm](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm) (states the monotonic log-likelihood-increase guarantee)

In [ ]:
expectation_maximization = __import__('8-EM').expectation_maximization

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
k = 4
pi, m, S, g, l = expectation_maximization(X, k, 150, verbose=True)
print(X.shape[0] * pi)
print(m)
print(S)
print(l)
clss = np.sum(g * np.arange(k).reshape(k, 1), axis=0)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(m[:, 0], m[:, 1], s=50, marker='*', c=list(range(k)))
plt.show()

## 9. Bayesian Information Criterion (BIC)

**What it does:** automates "how many clusters should my GMM have?", the same way task 3 did for K-means — but using a statistically principled score (BIC) instead of an eyeballed elbow.

**The formula:**

$$\text{BIC} = p\ln(n) - 2\ln(\hat L)$$

where $p$ is the number of free parameters in the model, $n$ is the number of data points, and $\hat L$ is the model's maximized likelihood (so $-2\ln(\hat L)$ is just `-2 * ll`, using the log-likelihood EM already computed). **Lower BIC is better** — it rewards a good fit but penalizes unnecessary complexity.

**How it works:**
1. For every candidate `k` from `kmin` to `kmax`: run the *entire* EM algorithm (task 8) from scratch to convergence.
2. Count that model's free parameters `p`: `k` means of size `d`, `k` priors (minus 1, since priors must sum to `1`), and `k` symmetric `d × d` covariance matrices (a symmetric matrix has $d(d+1)/2$ independent entries, not $d^2$).
3. Compute BIC with the formula above.
4. Pick the `k` with the **lowest** BIC as `best_k`.

**Key lines explained:**
```python
p = k * (1 + d + d * (d + 1) // 2) - 1
bic[i] = p * np.log(n) - 2 * ll
best_idx = np.argmin(bic)
best_k = kmin + best_idx
```
- `1 + d + d * (d + 1) // 2`: per cluster, that's `1` (its prior) + `d` (its mean vector) + $d(d+1)/2$ (its unique covariance entries).
- The trailing `- 1`: subtracts exactly one degree of freedom *overall*, because the `k` priors are constrained to sum to `1` — once `k - 1` of them are known, the last is fully determined, so it isn't a truly "free" parameter.
- `np.argmin`: returns the *position* of the smallest BIC in the array, not the value itself — that position is translated back into an actual `k` with `kmin + best_idx`.

**Where this fits in the pipeline:** this is the GMM pipeline's finish line, mirroring task 3's role for K-means — both answer "what's the best `k`?", starting from very different underlying models (hard variance-minimization here vs. probabilistic likelihood there).

> **Professor's note:** BIC has a well-known sibling, **AIC** (Akaike Information Criterion) — the only difference is the penalty term ($k\ln n$ for BIC vs. $2k$ for AIC). That difference matters: BIC penalizes extra parameters more heavily as the dataset grows, so it tends to prefer *simpler* models than AIC on large datasets. `scikit-learn`'s `GaussianMixture` (task 11) conveniently exposes both via `.bic()` and `.aic()`, so you can compare them directly.

**References:**
- [Wikipedia — Bayesian information criterion](https://en.wikipedia.org/wiki/Bayesian_information_criterion)

In [ ]:
BIC = __import__('9-BIC').BIC

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
best_k, best_result, l, b_ic = BIC(X, kmin=1, kmax=10)
print(best_k)
print(best_result)
print(l)
print(b_ic)
plt.plot(range(1, 11), l, 'r-', label='Log Likelihood')
plt.xlabel('Clusters')
plt.legend()
plt.show()
plt.plot(range(1, 11), b_ic, 'b-', label='BIC')
plt.xlabel('Clusters')
plt.legend()
plt.show()

## 10. K-means (scikit-learn)

**What it does:** the same algorithm conceptually as task 1, but using `scikit-learn`'s optimized, production-grade implementation instead of the from-scratch NumPy version.

**How it works:**
1. Instantiate `sklearn.cluster.KMeans(n_clusters=k)`.
2. `.fit(X)`: runs the whole Lloyd's-algorithm loop internally — using the smarter `k-means++` initialization by default (instead of task 0's pure uniform random guess), multiple random restarts, and a fast compiled implementation.
3. Read off `.cluster_centers_` (equivalent to `C` from task 1) and `.labels_` (equivalent to `clss`).

**Key lines explained:**
```python
model = sklearn.cluster.KMeans(n_clusters=k)
model.fit(X)
return model.cluster_centers_, model.labels_
```
This follows the standard `scikit-learn` **estimator pattern**, used consistently across the entire library: build an estimator object first (`KMeans(...)`), configuring hyperparameters at construction time; call `.fit(X)` to actually run the algorithm; then read the results off attributes that end in a trailing underscore (`cluster_centers_`, `labels_`) — a naming convention meaning "this was computed by `fit()`, not provided by you." The exact same pattern reappears in task 11's `GaussianMixture`.

**Where this fits in the pipeline:** a drop-in, production-quality replacement for task 1, now that tasks 0–9 have shown you what's actually happening under the hood.

> **Professor's note:** this is an important lesson in ML engineering: hand-rolling K-means (tasks 0–1) is invaluable for *understanding* the algorithm, but in real projects you should almost always reach for the maintained, tested, optimized library implementation. `scikit-learn`'s version defaults to smarter initialization (`k-means++`), handles edge cases you might not have thought of, and runs far more optimized code than a first NumPy draft.

**References:**
- [scikit-learn — `sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) (official docs — see `init`, `n_init`, and the note on Lloyd's/Elkan's algorithm)

In [ ]:
kmeans_sklearn = __import__('10-kmeans').kmeans

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
C, clss = kmeans_sklearn(X, 5)
print(C)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(C[:, 0], C[:, 1], s=50, marker='*', c=list(range(5)))
plt.show()

## 11. Gaussian Mixture Model (scikit-learn)

**What it does:** the same conceptual model as tasks 4–8 (a GMM fit via EM), but using `scikit-learn`'s `GaussianMixture`, which also directly reports the BIC score.

**How it works:**
1. Instantiate `sklearn.mixture.GaussianMixture(n_components=k)`.
2. `.fit(X)` runs EM internally to convergence.
3. `.predict(X)` performs a *hard* assignment per point (the cluster with the highest responsibility) — handy for coloring a scatter plot, even though the underlying model is fully probabilistic.
4. `.bic(X)` computes the Bayesian Information Criterion for the fitted model directly — no need to hand-roll task 9's `p * log(n) - 2 * ll` formula yourself.

**Key lines explained:**
```python
model = sklearn.mixture.GaussianMixture(n_components=k)
model.fit(X)
clss = model.predict(X)
bic = model.bic(X)
return model.weights_, model.means_, model.covariances_, clss, bic
```
- `n_components` is `scikit-learn`'s name for what tasks 4–9 called `k` — same idea, different vocabulary. This exact mapping (`k` ↔ `pi`/`m`/`S` ↔ `weights_`/`means_`/`covariances_`) is worth memorizing, since it's how you'll translate between "from scratch" ML code and library code throughout your career.
- `model.weights_`, `.means_`, `.covariances_`: the fitted `pi`, `m`, `S` respectively — again with the trailing-underscore "computed by `fit()`" convention from task 10.

**Where this fits in the pipeline:** the `scikit-learn` equivalent of the entire tasks 4–9 block — everything you built by hand (initialization, E-step, M-step, EM loop, BIC search) is exactly what `GaussianMixture.fit()` + `.bic()` do internally, in two lines.

> **Professor's note:** `scikit-learn`'s `covariance_type` parameter (default `'full'` — each cluster gets its own general covariance matrix, exactly what our from-scratch `S` represents) is worth exploring. `'diag'` or `'spherical'` covariance types use fewer parameters per cluster (directly lowering BIC's complexity penalty from task 9) at the cost of only being able to fit axis-aligned or perfectly round clusters — a classic bias/variance trade-off you'll see again and again in ML.

**References:**
- [scikit-learn — `sklearn.mixture.GaussianMixture`](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) (official docs — `n_components`, `.bic()`, `covariance_type`)

In [ ]:
gmm = __import__('11-gmm').gmm

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S, clss, bic = gmm(X, 4)
print(pi)
print(m)
print(S)
print(bic)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(m[:, 0], m[:, 1], s=50, marker='*', c=list(range(4)))
plt.show()

## 12. Agglomerative (hierarchical) clustering

**What it does:** a fundamentally different clustering strategy from K-means/GMM. Instead of picking `k` up front and iterating, agglomerative clustering starts with *every point as its own cluster* and repeatedly merges the two closest clusters until only one remains, recording the whole merge history as a tree (a **dendrogram**). You then "cut" that tree at some height to get your final clusters.

**How it works** (using **Ward linkage** specifically):
1. Start: `n` clusters, one per data point.
2. Find the pair of clusters whose merge would increase the total within-cluster variance the *least* — this is exactly what Ward linkage optimizes, in contrast to other linkage criteria (e.g. "single" or "complete" linkage) that look at raw point-to-point distances instead of variance.
3. Merge that pair into one cluster; record the distance ("height") at which they merged.
4. Repeat steps 2–3 until only one cluster (containing every point) remains — this produces the full dendrogram.
5. To get actual, flat cluster labels, "cut" the dendrogram at a chosen height/distance (`dist`): every branch still separate at that height becomes its own final cluster.

**Key lines explained:**
```python
linkage = scipy.cluster.hierarchy.linkage(X, method='ward')
clss = scipy.cluster.hierarchy.fcluster(linkage, t=dist, criterion='distance')
scipy.cluster.hierarchy.dendrogram(linkage, color_threshold=dist)
plt.show()
```
- `linkage(...)` performs all the merging described above and returns an `(n-1, 4)` matrix — one row per merge — recording which two clusters merged, at what distance, and how many points the resulting cluster now contains.
- `fcluster(..., t=dist, criterion='distance')` "cuts" the tree at height `dist`, converting the merge history into flat integer cluster labels — exactly like the `clss` arrays returned by every earlier task.
- `dendrogram(..., color_threshold=dist)` is purely visual: it draws the tree and colors branches below `dist` differently per cluster, so the plotted colors visually match the `dist` cut used by `fcluster`.

**Where this fits in the pipeline:** a structurally different alternative to everything else in this notebook. Where K-means and GMMs need you to *commit* to a number of clusters `k` up front (exactly the hard problem tasks 3 and 9 solve), agglomerative clustering instead asks you to pick a *distance threshold* (`dist`) and lets the merge history determine how many clusters that implies — and, unlike the other two families, its dendrogram gives you a visual map of the cluster structure at *every possible threshold at once*, not just the one you chose.

> **Professor's note:** the "maximum cophenetic distance for all clusters" (the `dist` argument) is exactly the y-axis height at which you visually "cut" a dendrogram plot with a horizontal line. Try re-running the cell below with a smaller or larger `dist` and watch both the plot and the number of resulting clusters change — it's one of the most intuitive hyperparameters to build intuition for by experimentation.

**References:**
- [SciPy — `scipy.cluster.hierarchy.linkage`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) (official docs — the Ward formula and the linkage matrix format)
- [Wikipedia — Ward's method](https://en.wikipedia.org/wiki/Ward%27s_method)
- [Wikipedia — Cophenetic correlation](https://en.wikipedia.org/wiki/Cophenetic_correlation) (defines cophenetic distance)

In [ ]:
agglomerative = __import__('12-agglomerative').agglomerative

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=100)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=100)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
clss = agglomerative(X, 100)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.show()